In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from nltk.corpus import brown
import nltk
import numpy as np
import random

# Ensure Brown corpus is downloaded
nltk.download('brown')
nltk.download('universal_tagset')

# CONFIG
MODEL_NAME = "huawei-noah/TinyBERT_General_4L_312D"
MAX_LEN, BATCH_SIZE, EPOCHS, LR, DROPOUT = 128, 32, 3, 2e-5, 0.3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# LOAD BROWN CORPUS
data = list(brown.tagged_sents(tagset='universal'))
random.shuffle(data)
data = data[:8000]  # 250 batches of 32

# CLEAN DATA TO MATCH TEXTS AND LABEL LENGTHS
cleaned_data = []
for sent in data:
    words = [w for w, t in sent]
    tags = [t for w, t in sent]
    if len(words) == len(tags) and len(words) > 0:
        cleaned_data.append((words, tags))

sentences = [x[0] for x in cleaned_data]
labels = [x[1] for x in cleaned_data]

# CREATE LABEL MAPPINGS
pos_labels = sorted(set(tag for seq in labels for tag in seq))
pos_label2id = {label: i for i, label in enumerate(pos_labels)}
pos_id2label = {i: label for label, i in pos_label2id.items()}
print("Detected POS labels:", pos_labels)

# ENCODE LABELS
encoded_labels = [[pos_label2id[tag] for tag in seq] for seq in labels]

# SPLIT DATA
train_texts, temp_texts, train_labels, temp_labels = train_test_split(sentences, encoded_labels, test_size=0.2, random_state=42)
val_texts, test_texts, val_labels, test_labels = train_test_split(temp_texts, temp_labels, test_size=0.5, random_state=42)

# TOKENIZER
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# DATASET CLASS
class POSDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts, self.labels = texts, labels
        self.tokenizer, self.max_len = tokenizer, max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        words = self.texts[idx]
        label_seq = self.labels[idx]
        encodings = self.tokenizer(words, is_split_into_words=True,
                                   return_offsets_mapping=True, padding='max_length',
                                   truncation=True, max_length=self.max_len)
        input_ids = torch.tensor(encodings['input_ids'])
        attention_mask = torch.tensor(encodings['attention_mask'])
        label_ids = torch.full((self.max_len,), -100)
        word_ids = encodings.word_ids()
        for i, word_idx in enumerate(word_ids):
            if word_idx is not None and word_idx < len(label_seq):
                label_ids[i] = label_seq[word_idx]
        return input_ids, attention_mask, label_ids

# CREATE SAMPLER FOR CLASS BALANCE (SEQUENCE-LEVEL)
dominant_labels = [max(set(seq), key=seq.count) for seq in train_labels]
class_sample_count = np.array([dominant_labels.count(i) for i in range(len(pos_label2id))])
weights = 1. / class_sample_count
sample_weights = [weights[label] for label in dominant_labels]
num_samples = 8000  # 250 batches of 32
sampler = WeightedRandomSampler(sample_weights, num_samples=num_samples, replacement=True)

# LOADERS
train_dataset = POSDataset(train_texts, train_labels, tokenizer, MAX_LEN)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
val_loader = DataLoader(POSDataset(val_texts, val_labels, tokenizer, MAX_LEN), batch_size=1)
test_loader = DataLoader(POSDataset(test_texts, test_labels, tokenizer, MAX_LEN), batch_size=1)

# GENERATOR
class Generator(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(DROPOUT)
        self.fc = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        output = self.bert(input_ids, attention_mask=attention_mask).last_hidden_state
        output = self.dropout(output)
        logits = self.fc(output)
        return logits

# DISCRIMINATOR
class Discriminator(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.hidden_size = self.bert.config.hidden_size + num_labels
        num_heads = 4
        while self.hidden_size % num_heads != 0:
            num_heads -= 1
        self.num_heads = max(1, num_heads)
        self.attention = nn.MultiheadAttention(embed_dim=self.hidden_size, num_heads=self.num_heads, batch_first=True)
        self.fc = nn.Linear(self.hidden_size, 2)

    def forward(self, input_ids, attention_mask, tag_logits):
        text_embeds = self.bert(input_ids, attention_mask=attention_mask).last_hidden_state
        tag_embeds = F.gumbel_softmax(tag_logits, tau=0.5, hard=False)
        combined = torch.cat([text_embeds, tag_embeds], dim=-1)
        combined = combined[:, :attention_mask.size(1), :]
        attn_output, _ = self.attention(combined, combined, combined)
        logits = self.fc(attn_output)
        return logits

# MODELS & OPTIMIZERS
G = Generator(MODEL_NAME, len(pos_label2id)).to(DEVICE)
D = Discriminator(MODEL_NAME, len(pos_label2id)).to(DEVICE)
optimizer_G = AdamW(G.parameters(), lr=LR)
optimizer_D = AdamW(D.parameters(), lr=LR)
ce_loss = nn.CrossEntropyLoss(ignore_index=-100)

# TRAINING LOOP
for epoch in range(EPOCHS):
    G.train(); D.train()
    for input_ids, attention_mask, labels in tqdm(train_loader, desc=f"[POS] Epoch {epoch+1}"):
        input_ids, attention_mask, labels = input_ids.to(DEVICE), attention_mask.to(DEVICE), labels.to(DEVICE)

        tag_logits = G(input_ids, attention_mask)
        g_loss = ce_loss(tag_logits.view(-1, len(pos_label2id)), labels.view(-1))

        d_logits = D(input_ids, attention_mask, tag_logits)
        real_targets = (labels != -100).long()
        d_loss = ce_loss(d_logits.view(-1, 2), real_targets.view(-1))

        optimizer_G.zero_grad(); optimizer_D.zero_grad()
        (g_loss + d_loss).backward()
        optimizer_G.step(); optimizer_D.step()

    print(f"[POS] Epoch {epoch+1} | G Loss: {g_loss.item():.4f} | D Loss: {d_loss.item():.4f}")

# EVALUATION

def evaluate_pos_generator(model, loader, name):
    print(f"\n--- {name} POS Evaluation ---")
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for i, (input_ids, attention_mask, labels) in enumerate(loader):
            input_ids, attention_mask = input_ids.to(DEVICE), attention_mask.to(DEVICE)
            logits = model(input_ids, attention_mask)
            preds = torch.argmax(logits, dim=-1).cpu().numpy()
            labels = labels.numpy()
            for p_seq, l_seq in zip(preds, labels):
                for p, l in zip(p_seq, l_seq):
                    if l != -100:
                        preds_all.append(pos_id2label[p])
                        labels_all.append(pos_id2label[l])
            if (i + 1) % 100 == 0:
                print(f"Processed {i+1}/{len(loader)} batches...")
    print(classification_report(labels_all, preds_all, digits=4))

evaluate_pos_generator(G, val_loader, "Validation")
evaluate_pos_generator(G, test_loader, "Test")


[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!


Detected POS labels: ['.', 'ADJ', 'ADP', 'ADV', 'CONJ', 'DET', 'NOUN', 'NUM', 'PRON', 'PRT', 'VERB', 'X']


[POS] Epoch 1: 100%|██████████| 250/250 [25:23<00:00,  6.09s/it]


[POS] Epoch 1 | G Loss: 0.1895 | D Loss: 0.0012


[POS] Epoch 2: 100%|██████████| 250/250 [25:14<00:00,  6.06s/it]


[POS] Epoch 2 | G Loss: 0.1668 | D Loss: 0.0003


[POS] Epoch 3: 100%|██████████| 250/250 [24:40<00:00,  5.92s/it]


[POS] Epoch 3 | G Loss: 0.0841 | D Loss: 0.0001

--- Validation POS Evaluation ---
Processed 100/800 batches...
Processed 200/800 batches...
Processed 300/800 batches...
Processed 400/800 batches...
Processed 500/800 batches...
Processed 600/800 batches...
Processed 700/800 batches...
Processed 800/800 batches...
              precision    recall  f1-score   support

           .     0.9910    0.9996    0.9953      2435
         ADJ     0.8134    0.8795    0.8452      1403
         ADP     0.9714    0.9831    0.9772      1898
         ADV     0.8993    0.9077    0.9035       856
        CONJ     0.9913    0.9913    0.9913       462
         DET     0.9918    0.9902    0.9910      1842
        NOUN     0.9614    0.9259    0.9433      4657
         NUM     0.7205    0.9880    0.8333       167
        PRON     0.9838    0.9653    0.9745       692
         PRT     0.9315    0.9077    0.9194       509
        VERB     0.9628    0.9532    0.9580      2799
           X     0.0000    0.0000   